<a href="https://colab.research.google.com/github/carlocamilloni/Structural-Bioinformatics/blob/main/Notebooks/report_6_MD_enhanced_martini.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# REPORT: Enhanced sampling and simplified models in MD

First save a copy of this notebook in your Google Drive (`File -> Save a copy in Drive`).

This is a group report, and as in the training notebook (`t06_MD_enhanced_martini`) the two groups work on different topics:

*   **GROUP 1: enhanced sampling.** The *cis/trans* isomerisation of prolyl peptide bonds, studied with well-tempered metadynamics.
*   **GROUP 2: simplified models.** The phase behaviour of lipid bilayers and the effect of cholesterol, studied with the Martini coarse-grained model.

The notebook has two parts for each group:

1. **Running the simulations**: the protocol of the training notebook collected in a few cells. Each member of the group runs one or more of the simulations.
2. **The report**: the analysis of all the simulations of the group, with your text. Write it in the text cells and add plots with the code cells; the cells with *your code here* are for you.

The cells that start with `%%group 1` or `%%group 2` run only for the group you choose below, so you can also use *Runtime -> Run all*.

**Group:** 1 / 2

**Members:**

In [ ]:
from datetime import date

today = date.today()
print(today)

## The assignment

### GROUP 1: *cis/trans* isomerisation of prolyl peptide bonds

The peptide bond has a partial double-bond character and is planar: its dihedral angle $\omega$ (CA-C-N-CA) is close to 180° (*trans*) or to 0° (*cis*). For most peptide bonds the *trans* form is so much more stable that *cis* bonds are almost never observed, except for the bonds that precede a proline (X-Pro bonds): the cyclic side chain of proline makes the two forms similar in energy. In short unstructured peptides in water a few to a few tens of percent of the X-Pro bonds are *cis*, depending on the residue X and on the neighbouring residues, and in folded proteins about 5% of the X-Pro bonds are *cis*. The barrier between the two forms is high (80-90 kJ/mol), so that the isomerisation takes seconds to minutes at room temperature: it is often the slowest step of protein folding, and it is catalysed in the cell by peptidyl-prolyl isomerases such as cyclophilins, FKBPs and Pin1. In T04, Group A met a prolyl bond that is *cis* in β2-microglobulin and *trans* in its amyloidogenic variant ΔN6.

In this task you study the model peptide Ace-X-Pro-NMe in water, where X is a residue of your choice, and compute with well-tempered metadynamics on $\omega$ the free-energy profile of the isomerisation.

**Questions:** What is the population of the *cis* form, and how does it depend on the residue that precedes the proline and on the force field? How do the simulations compare with the experiments? How high is the barrier, and how long would you need to wait to observe an isomerisation in a standard MD simulation? Does the conformation of the residue that precedes the proline depend on the isomer?

### GROUP 2: phase behaviour of lipid bilayers

Lipid bilayers made of a single kind of saturated lipid, such as DPPC, undergo a sharp *main transition* when they are cooled: from a fluid (liquid-disordered) phase, where the tails are disordered and the lipids diffuse quickly, to a gel phase, where the tails are extended, ordered and tilted, and the lipids are almost immobile. For DPPC the experimental transition temperature $T_m$ is 314 K (41 °C). Cholesterol changes these properties dramatically: at high concentration it removes the transition, and it orders the tails of the fluid phase (*condensing effect*) forming the liquid-ordered phase, which is believed to be at the origin of the *rafts* of cell membranes. Phase transitions are a severe test of a force field and of a coarse-grained model in particular: the recent refinement of the Martini 3 lipids aimed, among other things, at improving their phase behaviour (Pedersen et al., 2025).

**Questions:** At which temperature does a DPPC bilayer become a gel in Martini? How do the area per lipid, the thickness, the order of the tails and the lateral diffusion change at the transition? Do you observe hysteresis between cooling and heating? How does cholesterol change the properties of a DPPC bilayer, above and below the transition? How do the results compare with the experiments?

## Organising the work

**GROUP 1.** Every member runs at least one system (one residue X and one force field). The group must include:

*   the reference system **Ace-Ala-Pro-NMe with AMBER99SB-ILDN**;
*   **at least three different residues X** with AMBER99SB-ILDN, chosen to test a hypothesis (e.g. aromatic vs aliphatic, small vs large, charged...): look in the literature for experimental *cis* populations of X-Pro bonds in short peptides before choosing;
*   **at least one more force field** (`amber14sb` with TIP3P water or `amber19sb` with OPC water) for Ala-Pro.

Each system needs a standard MD run (10 ns) and a metadynamics run (20 ns by default). The system has about 3000 atoms: on two CPU cores GROMACS runs about 100 ns/day (5 hours for 20 ns), on a Colab GPU it is several times faster. Look at the performance printed at the end of the standard MD run to plan your time, and start early.

**GROUP 2.** Every member runs some of the simulations of the following two series:

*   **temperature series**: pure DPPC between 300 and 335 K (at least six temperatures, closer around the transition), all started from a new fluid-like bilayer built by `insane` (cooling), plus at least two runs started from the final structure of a gel (heating);
*   **cholesterol series**: DPPC with 0, 10, 20, 30 and 40% cholesterol at 323 K, and the same at one temperature where pure DPPC is a gel.

Optional: DOPC or POPC, which have unsaturated tails, at the same temperatures. The default runs (about 6000 beads, 100 ns) take a few hours on two CPU cores and much less on a GPU; near the transition the gel can take longer to form, so look at the time series before deciding that a run is finished.

To share the results, one member creates a folder in Google Drive (e.g. `Task6_group1`) and shares it with the rest of the group, with editing rights. Every member adds a shortcut to it in his or her own `MyDrive` (right click on the folder -> `Organise` -> `Add shortcut`). Each simulation is written in its own sub-folder, so different members never write the same files. If you run the simulations on your own computer, they are written in the folder `T06_group` next to this notebook.

# Setup (run these cells at the start of every session)

In [ ]:
GROUP = 1      # 1 (prolyl isomerisation) or 2 (lipid bilayers)

In [ ]:
!nvidia-smi || echo "no NVIDIA GPU found"

In [ ]:
%%bash
# install GROMACS 2026.3 with CUDA (GPU) support and PLUMED 2.10 from conda-forge, unless they are already installed
# (see the training notebook t06_MD_enhanced_martini for details)
if ! command -v gmx > /dev/null || ! command -v plumed > /dev/null; then
  cd /content
  curl -sL -o micromamba https://github.com/mamba-org/micromamba-releases/releases/latest/download/micromamba-linux-64
  chmod +x micromamba
  ./micromamba create -y -q -r /content/mamba -p /content/gmx -c conda-forge "gromacs=2026.3=nompi_cuda*" "plumed=2.10"
  ln -sf /content/gmx/bin/gmx /usr/local/bin/gmx
  ln -sf /content/gmx/bin/plumed /usr/local/bin/plumed
fi
gmx --version | grep -E "GROMACS version|GPU support|Plumed support"
echo "PLUMED version: $(plumed info --version)"

In [ ]:
import os, glob, shutil
# GROMACS finds PLUMED through PLUMED_KERNEL, the path of the PLUMED library
plumed_bin = os.path.dirname(os.path.realpath(shutil.which('plumed')))
os.environ['PLUMED_KERNEL'] = os.path.abspath(glob.glob(os.path.join(plumed_bin, '..', 'lib', 'libplumedKernel.*'))[0])

# MDAnalysis for the analysis, py3Dmol for visualisation, PeptideBuilder to build the peptides of group 1
%pip install -q MDAnalysis py3Dmol PeptideBuilder

In [ ]:
import os, sys
if 'BASE_DIR' not in globals():          # remember where we started: running this cell again does not go one folder deeper
    BASE_DIR = os.getcwd()

# on Colab we work in the folder shared by the group on Google Drive (see above);
# on your own computer in a folder next to this notebook
use_drive = 'google.colab' in sys.modules
if use_drive:
    from google.colab import drive
    drive.mount('/content/drive')
    local_path = f'/content/drive/MyDrive/Task6_group{GROUP}'
else:
    local_path = os.path.join(BASE_DIR, 'T06_group')
os.makedirs(local_path, exist_ok=True)
%cd $local_path

In [ ]:
# the input files are in the Data folder of the course repository:
# if this notebook is run from a copy of the repository we use it, otherwise we download the repository
if os.path.isdir(os.path.join(BASE_DIR, '..', 'Data', 'md', 'metad')):
    REPO_DIR = os.path.abspath(os.path.join(BASE_DIR, '..'))
else:
    REPO_DIR = os.path.join(BASE_DIR, 'sb')
    if os.path.isdir(REPO_DIR):
        !cd "$REPO_DIR" && git pull -q
    else:
        !git clone -q --depth 1 https://github.com/carlocamilloni/Structural-Bioinformatics.git "$REPO_DIR"
METAD_DIR = os.path.join(REPO_DIR, 'Data', 'md', 'metad')
MARTINI_DIR = os.path.join(REPO_DIR, 'Data', 'martini')
print('input files from', METAD_DIR, 'and', MARTINI_DIR)

In [ ]:
import glob, subprocess, warnings
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
warnings.filterwarnings('ignore')

import linecache
from IPython import get_ipython
from IPython.core.magic import register_cell_magic

@register_cell_magic
def group(line, cell):
    """A cell that starts with %%group N runs only if GROUP is N."""
    ipython = get_ipython()
    if int(line) != ipython.user_ns.get('GROUP'):
        print(f'This cell is for group {line}: skipped.')
        return
    code = ipython.transform_cell(cell)
    filename = f'<group {line} cell>'
    linecache.cache[filename] = (len(code), None, code.splitlines(True), filename)   # so that errors show the code
    exec(compile(code, filename, 'exec'), ipython.user_ns)

def read_xvg(filename):
    """Read a GROMACS .xvg file. Returns (data, legends): data is a 2D array whose first column is the x axis."""
    rows, legends = [], []
    with open(filename) as f:
        for line in f:
            if line.startswith('@') and 'legend "' in line and line.split()[1].startswith('s'):
                legends.append(line.split('"')[1])
            elif not line.startswith(('#', '@')) and line.strip():
                rows.append([float(x) for x in line.split()])
    return np.array(rows), legends

def read_plumed(filename):
    """Read a PLUMED output file (COLVAR, HILLS, fes). Returns a dictionary {column name: numpy array}."""
    with open(filename) as f:
        fields = f.readline().split()[2:]
    data = np.loadtxt(filename, comments='#', ndmin=2)
    return {name: data[:, i] for i, name in enumerate(fields)}

kT = 0.0083144626 * 300      # kJ/mol at 300 K

# GROUP 1, Part 1: running the simulations

Set the parameters of your system in the next cell and then run the cells below in order.

The protocol: the peptide Ace-X-Pro-NMe is built in an extended conformation with *trans* peptide bonds, placed in a rhombic dodecahedron box with 1.2 nm of water around it, minimised and equilibrated for 500 ps at 300 K and 1 bar. Then a standard MD run and a well-tempered metadynamics run, both starting from the equilibrated structure. The metadynamics biases $\omega$ of the X-Pro bond with Gaussians of initial height 1.2 kJ/mol and width 0.15 rad every 1 ps, and a bias factor of 10.

**If Colab disconnects** during a run: run the setup cells again, then the parameters cell **with the same values**, and then the cell of the run that was interrupted. The simulation continues from the last checkpoint.

In [ ]:
%%group 1
# ---- edit these values for each system ------------------------------------------------
X           = 'A'                 # one-letter code of the residue before the proline: Ace-X-Pro-NMe
FORCE_FIELD = 'amber99sb-ildn'    # amber99sb-ildn, amber14sb or amber19sb
MD_NS       = 10                  # length of the standard MD run (ns)
MTD_NS      = 20                  # length of the metadynamics run (ns)
REBUILD     = 'no'                # 'yes' to delete this system and prepare it again from scratch
# ----------------------------------------------------------------------------------------

WATER = {'amber99sb-ildn': ('tip3p', 'spc216.gro'), 'amber14sb': ('tip3p', 'spc216.gro'),
         'amber19sb': ('opc', 'tip4p.gro')}       # recommended water model and box of water to solvate
SYSTEM = f'{X}P_{FORCE_FIELD}'
run_dir = f'{local_path}/{SYSTEM}'
os.makedirs(run_dir, exist_ok=True)
os.environ.update(RUN_DIR=run_dir, METAD_DIR=METAD_DIR, FF=FORCE_FIELD, WATER=WATER[FORCE_FIELD][0],
                  WATER_BOX=WATER[FORCE_FIELD][1], REBUILD=REBUILD,
                  MD_STEPS=str(int(MD_NS * 500000)), MTD_STEPS=str(int(MTD_NS * 500000)))    # time step of 2 fs
print('system', SYSTEM, 'in', run_dir)

### Building, minimising and equilibrating the system

The peptide is built with [PeptideBuilder](https://github.com/clauswilke/PeptideBuilder): we build Gly-X-Pro-Gly and turn the first and the last glycine into the acetyl and N-methyl caps. Then the cell writes the output of each program to a `.log` file and prints only a short summary; if something goes wrong it shows the end of the log of the step that failed.

In [ ]:
%%group 1
import io
import PeptideBuilder
from PeptideBuilder import Geometry
from Bio.PDB import PDBIO

def build_capped_peptide(sequence, filename, nme_carbon='CH3'):
    """Write a PDB file of Ace-sequence-NMe in an extended conformation, with trans peptide bonds.
    nme_carbon is the name of the carbon of the N-methyl cap, which depends on the force field."""
    structure = PeptideBuilder.initialize_res('G')
    for aa in sequence + 'G':
        geometry = Geometry.geometry(aa)
        geometry.phi, geometry.psi_im1 = -75, 145
        PeptideBuilder.add_residue(structure, geometry)
    pdb = io.StringIO()
    writer = PDBIO()
    writer.set_structure(structure)
    writer.save(pdb)
    lines = [l for l in pdb.getvalue().splitlines() if l.startswith('ATOM')]
    first, last = lines[0][22:26], lines[-1][22:26]
    out = []
    for l in lines:
        name, resid = l[12:16].strip(), l[22:26]
        if resid == first:            # glycine -> acetyl: keep CA (renamed CH3), C and O
            if name == 'N':
                continue
            l = l[:17] + 'ACE' + l[20:]
        elif resid == last:           # glycine -> N-methyl: keep N and CA (renamed CH3)
            if name in ('C', 'O'):
                continue
            l = l[:17] + 'NME' + l[20:]
        if resid == first and name == 'CA':
            l = l[:12] + ' CH3' + l[16:]
        if resid == last and name == 'CA':
            l = l[:12] + f' {nme_carbon:<3s}' + l[16:]
        out.append(l)
    with open(filename, 'w') as f:
        f.write('\n'.join(out) + '\nEND\n')

if REBUILD == 'yes' or not os.path.exists(f'{run_dir}/npt.gro'):
    build_capped_peptide(X + 'P', f'{run_dir}/peptide.pdb', 'C' if FORCE_FIELD in ('amber14sb', 'amber19sb') else 'CH3')

In [ ]:
%%group 1
%%bash
cd "$RUN_DIR"
# run a command, save its output in a log file and stop showing the log if it fails
step() { local log=$1; shift; if ! "$@" > "$log" 2>&1; then echo "ERROR: last lines of $log"; tail -25 "$log"; exit 1; fi; }

if [ "$REBUILD" = "yes" ]; then rm -f md.* mtd.* npt.* COLVAR* HILLS* bck.* \#*; fi
if [ -f npt.gro ]; then
  echo "This system has already been prepared: go to the production cells."
  echo "To start it again from scratch set REBUILD = 'yes' in the parameters cell."
  exit 0
fi
cp $METAD_DIR/em-water.mdp $METAD_DIR/npt-water.mdp $METAD_DIR/md-water.mdp .

step pdb2gmx.log gmx pdb2gmx -f peptide.pdb -o peptide.gro -ignh -ff $FF -water $WATER
step editconf.log gmx editconf -f peptide.gro -bt dodecahedron -d 1.2 -o box.gro
step solvate.log gmx solvate -cp box.gro -cs $WATER_BOX -p topol.top -o solv.gro
# if X is charged, one ion neutralises the system (grompp warns that the system is not neutral: -maxwarn 1)
step grompp_ions.log gmx grompp -f em-water.mdp -c solv.gro -p topol.top -o ions.tpr -maxwarn 1
echo SOL | step genion.log gmx genion -s ions.tpr -neutral -p topol.top -o ions.gro
sed -n '/molecules/,$p' topol.top
step grompp_em.log gmx grompp -f em-water.mdp -c ions.gro -p topol.top -o em.tpr
step em.log gmx mdrun -deffnm em
grep -E "Potential Energy|Maximum force" em.log
step grompp_npt.log gmx grompp -f npt-water.mdp -c em.gro -p topol.top -o npt.tpr
step npt.log gmx mdrun -deffnm npt
printf "Temperature\nPressure\nDensity\n\n" | step energy.log gmx energy -f npt.edr -o npt_energy.xvg
grep -E "^(Temperature|Pressure|Density)" energy.log
# reference structure for the PLUMED shortcuts (atom names and residue numbers)
step reference.log gmx editconf -f em.gro -o reference.pdb
echo "Done: system ready"

### The PLUMED inputs

PLUMED computes $\omega$ of the X-Pro bond (`@omega-3`, the bond between residue 2, X, and residue 3, Pro) and the backbone angles $\phi$ and $\psi$ of X and Pro. The same CVs are used to analyse the standard MD (with `plumed driver`) and to bias the metadynamics. When `mdrun` continues from a checkpoint (`-cpi`), GROMACS tells PLUMED to restart too: PLUMED reads the Gaussians already deposited in the HILLS file and appends to its output files instead of starting from scratch (look for `Restarting from HILLS` in `PLUMED.OUT`).

In [ ]:
%%group 1
CVS = """MOLINFO STRUCTURE=reference.pdb
omega: TORSION ATOMS=@omega-3
phi_x: TORSION ATOMS=@phi-2
psi_x: TORSION ATOMS=@psi-2
phi_p: TORSION ATOMS=@phi-3
psi_p: TORSION ATOMS=@psi-3
"""
with open(f'{run_dir}/plumed-cv.dat', 'w') as f:
    f.write(CVS + 'PRINT ARG=omega,phi_x,psi_x,phi_p,psi_p FILE=COLVAR-MD STRIDE=1\n')

with open(f'{run_dir}/plumed-mtd.dat', 'w') as f:
    f.write(CVS + """
metad: METAD ARG=omega PACE=500 HEIGHT=1.2 SIGMA=0.15 BIASFACTOR=10 TEMP=300 GRID_MIN=-pi GRID_MAX=pi FILE=HILLS CALC_RCT
PRINT ARG=omega,phi_x,psi_x,phi_p,psi_p,metad.bias,metad.rbias FILE=COLVAR-MTD STRIDE=500
""")
!cat {run_dir}/plumed-mtd.dat

### Standard MD

In [ ]:
%%group 1
%cd {run_dir}
if not os.path.exists('md.tpr'):
    !sed -i "s/^nsteps .*/nsteps = $MD_STEPS/" md-water.mdp
    !gmx grompp -f md-water.mdp -c npt.gro -t npt.cpt -p topol.top -o md.tpr > grompp_md.log 2>&1
!gmx mdrun -deffnm md -cpi md.cpt -cpt 5 -v
!plumed driver --plumed plumed-cv.dat --mf_xtc md.xtc > driver.log 2>&1 && echo "COLVAR-MD written"

### Metadynamics

In [ ]:
%%group 1
%cd {run_dir}
if not os.path.exists('mtd.tpr'):
    !sed "s/^nsteps .*/nsteps = $MTD_STEPS/" md-water.mdp > mtd-water.mdp
    !gmx grompp -f mtd-water.mdp -c npt.gro -t npt.cpt -p topol.top -o mtd.tpr > grompp_mtd.log 2>&1
!gmx mdrun -deffnm mtd -plumed plumed-mtd.dat -ntmpi 1 -cpi mtd.cpt -cpt 5 -v

In [ ]:
%%group 1
# a quick look at this run
mtd = read_plumed(f'{run_dir}/COLVAR-MTD')
fig, axes = plt.subplots(1, 2, figsize=(16, 4))
axes[0].plot(mtd['time'] / 1000, mtd['omega'], '.', ms=2)
axes[0].set_ylabel('$\\omega$ (rad)')
axes[1].plot(mtd['time'] / 1000, mtd['metad.bias'], '.', ms=2)
axes[1].set_ylabel('Bias (kJ/mol)')
for ax in axes:
    ax.set_xlabel('Time (ns)')
fig.suptitle(SYSTEM)
plt.show()

# GROUP 1, Part 2: the report

The analysis cells read all the systems that are in the group folder, so they can be run by any member once the simulations are finished.

## 1. Introduction

*Describe prolyl cis/trans isomerisation, its role in protein folding and function and what is known from experiments (populations, barriers, rates, the effect of the residue before the proline). Which systems did you choose and why? What do you expect?*

## 2. Methods

*Describe the systems and the simulation protocol (force fields, water models, box, temperature, length of the runs), the CVs, the metadynamics parameters and how you computed free energies, populations and their uncertainties.*

In [ ]:
%%group 1
# all the systems in the group folder: {name: folder}
systems = {os.path.basename(d): d for d in sorted(glob.glob(f'{local_path}/*P_*')) if os.path.exists(f'{d}/COLVAR-MTD')}
print('systems with a metadynamics run:', list(systems))
colors = dict(zip(systems, plt.cm.tab10.colors))

## 3. Results

### 3.1 Standard MD

*Does $\omega$ ever change in the standard MD runs? How large are its fluctuations? Which regions of the Ramachandran plot do X and Pro visit?*

In [ ]:
%%group 1
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for name, d in systems.items():
    if not os.path.exists(f'{d}/COLVAR-MD'):
        continue
    cv = read_plumed(f'{d}/COLVAR-MD')
    axes[0].plot(cv['time'] / 1000, cv['omega'], '.', ms=1, color=colors[name], label=name)
    axes[1].plot(cv['phi_x'], cv['psi_x'], '.', ms=1, color=colors[name])
    axes[2].plot(cv['phi_p'], cv['psi_p'], '.', ms=1, color=colors[name])
axes[0].set_xlabel('Time (ns)')
axes[0].set_ylabel('$\\omega$ (rad)')
axes[0].legend(markerscale=10)
for ax, res in zip(axes[1:], ['X', 'Pro']):
    ax.set_xlabel(f'$\\phi$ {res} (rad)')
    ax.set_ylabel(f'$\\psi$ {res} (rad)')
    ax.set_xlim(-np.pi, np.pi)
    ax.set_ylim(-np.pi, np.pi)
plt.show()

### 3.2 Free-energy profiles and their convergence

`sum_hills` reconstructs the free energy as a function of $\omega$ from the Gaussians, every 2 ns. We define *cis* as $|\omega|<\pi/2$ and *trans* as $|\omega|>\pi/2$ and compute for each estimate the free-energy difference $\Delta G = G_{cis} - G_{trans}$, the population of *cis* and the barrier. There are two transition states, around $\omega = \pm\pi/2$: the barrier is the lower of the two maxima, measured from the minimum of *trans*.

*Do the free-energy estimates converge? After how long? How many transitions between cis and trans did each simulation make?*

In [ ]:
%%group 1
def fes_series(d, stride=2000):
    """Run sum_hills on the HILLS file of folder d. Returns a list of (time in ns, omega, free energy)."""
    out = f'{d}/fes_'
    for f in glob.glob(out + '*.dat'):
        os.remove(f)
    subprocess.run(['plumed', 'sum_hills', '--hills', f'{d}/HILLS', '--mintozero', '--outfile', out,
                    '--stride', str(stride), '--kt', str(kT), '--bin', '360'], check=True, capture_output=True)
    times = read_plumed(f'{d}/HILLS')['time']
    files = sorted(glob.glob(out + '*.dat'), key=lambda f: int(f[len(out):-4]))
    series = []
    for i, f in enumerate(files):
        t = times[min((i + 1) * stride, len(times)) - 1] / 1000
        if not series or t > series[-1][0]:
            fes = read_plumed(f)
            series.append((t, fes['omega'], fes['file.free']))
    return series

def isomer_stats(omega, fes):
    """Delta G (cis - trans), cis population and barrier (kJ/mol) from a free-energy profile."""
    p = np.exp(-fes / kT)
    cis = np.abs(omega) < np.pi / 2
    p_cis = p[cis].sum() / p.sum()
    g_trans = fes[~cis].min()
    ts_plus = fes[(omega > np.pi / 4) & (omega < 3 * np.pi / 4)].max()
    ts_minus = fes[(omega < -np.pi / 4) & (omega > -3 * np.pi / 4)].max()
    return -kT * np.log(p_cis / (1 - p_cis)), p_cis, min(ts_plus, ts_minus) - g_trans

fes = {name: fes_series(d) for name, d in systems.items()}

fig, axes = plt.subplots(1, 3, figsize=(20, 5))
for name, series in fes.items():
    t, omega, f = series[-1]
    axes[0].plot(np.degrees(omega), f, color=colors[name], label=name)
    stats = np.array([isomer_stats(o, g) for _, o, g in series])
    times = [s[0] for s in series]
    axes[1].plot(times, stats[:, 0], 'o-', color=colors[name])
    axes[2].plot(times, stats[:, 2], 'o-', color=colors[name])
axes[0].set_xlabel('$\\omega$ (degrees)')
axes[0].set_ylabel('Free energy (kJ/mol)')
axes[0].legend()
axes[1].set_ylabel('$\\Delta G_{cis-trans}$ (kJ/mol)')
axes[2].set_ylabel('Barrier from trans (kJ/mol)')
for ax in axes[1:]:
    ax.set_xlabel('Time (ns)')
plt.show()

In [ ]:
%%group 1
# omega along the metadynamics runs: count the transitions
def count_transitions(omega):
    """Number of transitions between cis (|omega| < 60 degrees) and trans (|omega| > 120 degrees).
    The frames in between are assigned to the last state visited, so that recrossings of the barrier are not counted."""
    state, n = None, 0
    for w in np.abs(omega):
        new = 'cis' if w < np.pi / 3 else 'trans' if w > 2 * np.pi / 3 else state
        if state is not None and new != state:
            n += 1
        state = new
    return n

fig, axes = plt.subplots(len(systems), 1, figsize=(14, 2.5 * len(systems)), sharex=True, squeeze=False)
for ax, (name, d) in zip(axes[:, 0], systems.items()):
    mtd = read_plumed(f'{d}/COLVAR-MTD')
    ax.plot(mtd['time'] / 1000, np.degrees(mtd['omega']), '.', ms=1, color=colors[name])
    ax.set_ylabel('$\\omega$ (deg)')
    ax.set_title(f'{name}: {count_transitions(mtd["omega"])} transitions', fontsize=10)
axes[-1, 0].set_xlabel('Time (ns)')
plt.tight_layout()
plt.show()

### 3.3 Populations by reweighting, with an uncertainty

The *cis* population can also be computed by reweighting the frames of the metadynamics run with $w \propto e^{(V-c(t))/k_BT}$ (`metad.rbias`), as in the training notebook. This gives a way to estimate the uncertainty: with **block averaging** the run is divided in $N$ blocks, the population is computed in each block, and the standard error is $\sigma/\sqrt{N}$, where $\sigma$ is the standard deviation among the blocks. Frames close in time are correlated, so the blocks must be long enough to be independent: the error grows with the block length until it reaches a plateau, which is the estimate of the uncertainty.

*Does the population from reweighting agree with that from the free-energy profile? What is the uncertainty? Is 20 ns enough?*

In [ ]:
%%group 1
SKIP_NS = 2      # initial part of the metadynamics excluded from the reweighting

def reweighted(d):
    """Frames of the metadynamics after SKIP_NS, with their weights."""
    mtd = read_plumed(f'{d}/COLVAR-MTD')
    keep = mtd['time'] > SKIP_NS * 1000
    frames = {k: v[keep] for k, v in mtd.items()}
    frames['weight'] = np.exp((frames['metad.rbias'] - frames['metad.rbias'].max()) / kT)
    return frames

def block_error(values, weights, n_blocks):
    """Weighted average of values and its block-averaging standard error."""
    blocks = np.array_split(np.arange(len(values)), n_blocks)
    averages = np.array([np.average(values[b], weights=weights[b]) for b in blocks])
    return np.average(values, weights=weights), averages.std(ddof=1) / np.sqrt(n_blocks)

fig, ax = plt.subplots(figsize=(8, 5))
rows = []
for name, d in systems.items():
    frames = reweighted(d)
    is_cis = (np.abs(frames['omega']) < np.pi / 2).astype(float)
    n_blocks = np.array([2, 3, 4, 5, 6, 8, 10, 15, 20])
    errors = [block_error(is_cis, frames['weight'], n)[1] for n in n_blocks]
    block_ns = (frames['time'][-1] - frames['time'][0]) / 1000 / n_blocks
    ax.plot(block_ns, errors, 'o-', color=colors[name], label=name)
    p_cis, error = block_error(is_cis, frames['weight'], 5)
    _, p_fes, barrier = isomer_stats(*fes[name][-1][1:])
    rows.append({'system': name, 'cis (reweighting)': p_cis, '± (5 blocks)': error, 'cis (free energy)': p_fes,
                 'ΔG cis-trans (kJ/mol)': -kT * np.log(p_cis / (1 - p_cis)), 'barrier (kJ/mol)': barrier})
ax.set_xlabel('Block length (ns)')
ax.set_ylabel('Standard error of the cis population')
ax.legend()
plt.show()
results = pd.DataFrame(rows)
display(results.round(3))

### 3.4 How long does an isomerisation take?

Transition-state theory (Eyring equation) relates the barrier to the rate: $k = \frac{k_BT}{h} e^{-\Delta G^\ddagger/k_BT}$, with $k_BT/h = 6.25\times10^{12}$ s$^{-1}$ at 300 K. It gives only an order of magnitude (the prefactor of real molecules is smaller), but it is enough for a comparison.

*Estimate the average time needed for a trans→cis and for a cis→trans isomerisation. Compare it with the length of your standard MD runs, with the time actually needed by the metadynamics runs, and with experiments. By how much did metadynamics accelerate the process?*

In [ ]:
%%group 1
# your code here

### 3.5 Does the conformation of X depend on the isomer?

With the weights we can compute the free energy as a function of any CV, also separately for the frames in which the bond is *cis* and *trans*. Here the Ramachandran plot of the residue X.

*How does the conformation of X change between cis and trans? Can you explain it (think about the steric clashes between X and the proline ring)? Try also the angles of the proline.*

In [ ]:
%%group 1
def plot_ramachandran(ax, phi, psi, weights, title):
    hist, xe, ye = np.histogram2d(phi, psi, bins=50, range=[[-np.pi, np.pi], [-np.pi, np.pi]], weights=weights, density=True)
    with np.errstate(divide='ignore'):
        f = -kT * np.log(hist)
    f -= f[np.isfinite(f)].min()
    c = ax.contourf(0.5 * (xe[1:] + xe[:-1]), 0.5 * (ye[1:] + ye[:-1]), np.clip(f, 0, 25).T, levels=np.arange(0, 26, 2.5))
    plt.colorbar(c, ax=ax, label='Free energy (kJ/mol)')
    ax.set_xlabel('$\\phi$ (rad)')
    ax.set_ylabel('$\\psi$ (rad)')
    ax.set_title(title)

name = list(systems)[0]               # change to look at another system
frames = reweighted(systems[name])
is_cis = np.abs(frames['omega']) < np.pi / 2
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, isomer, sel in [(axes[0], 'trans', ~is_cis), (axes[1], 'cis', is_cis)]:
    plot_ramachandran(ax, frames['phi_x'][sel], frames['psi_x'][sel], frames['weight'][sel], f'{name}: X, {isomer}')
plt.show()

### 3.6 Comparison between systems and with experiments

*Collect in a table the cis populations, the free-energy differences and the barriers of all the systems, with their uncertainties, and the experimental values you found in the literature (give the references, and note the experimental conditions and the peptides used, which may differ from yours). Is the effect of the residue X reproduced? Which force field agrees best with the experiments?*

In [ ]:
%%group 1
# your code here

## 4. Discussion

*Interpret the results. Some points to consider: Are the differences between systems larger than the uncertainties? Is ω a good CV, or are there other slow degrees of freedom (look at the free-energy estimates as a function of time and at the other angles)? What are the limits of a simple model peptide to understand prolyl isomerisation in proteins? Which simulations would you add?*

## 5. Conclusions

## References

*   Barducci A, Bussi G, Parrinello M (2008) Well-tempered metadynamics: a smoothly converging and tunable free-energy method. *Phys Rev Lett* 100:020603
*   Tiwary P, Parrinello M (2015) A time-independent free energy estimator for metadynamics. *J Phys Chem B* 119:736
*   *add the experimental papers you used*

# GROUP 2, Part 1: running the simulations

Set the parameters of your simulation in the next cell and then run the cells below in order.

The protocol is that of the training notebook: the bilayer is built with `insane` with the composition you choose (the same in the two leaflets), minimised, equilibrated for 1 ns with a time step of 10 fs, and simulated at the chosen temperature and at zero surface tension (semi-isotropic pressure coupling at 1 bar) with a time step of 20 fs. To heat a gel instead, set `START_FROM` to the name of a finished simulation: its final structure is used as the starting point.

The default size of the box in the plane of the membrane, 7 nm, gives about 80 lipids per leaflet; the box is 12 nm high, which gives more than 20 water beads per lipid (with less water, the thin layer of Martini water tends to freeze, see below). All the simulations of a series should have the same size and length.

**If Colab disconnects** during a run: run the setup cells again, then the parameters cell **with the same values**, and then the production cell. The simulation continues from the last checkpoint.

In [ ]:
%%group 2
# ---- edit these values for each simulation -------------------------------------------------
LIPIDS     = 'DPPC:1'     # lipids and relative amounts, as for insane, e.g. 'DPPC:1', 'DPPC:9 CHOL:1', 'DPPC:7 CHOL:3', 'DOPC:1'
T          = 323          # temperature (K)
SIM_NS     = 100          # length of the production run (ns)
START_FROM = ''           # '' to build a new bilayer, or the name of a finished simulation to start from its last frame
BOX_XY     = 7            # size of the box in the plane of the membrane (nm)
REBUILD    = 'no'         # 'yes' to delete this simulation and prepare it again from scratch
# --------------------------------------------------------------------------------------------

composition = '_'.join(item.replace(':', '') for item in LIPIDS.split())
SIM = f'{composition}_{T}K' + (f'_from_{START_FROM}' if START_FROM else '')
run_dir = f'{local_path}/{SIM}'
os.makedirs(run_dir, exist_ok=True)
os.environ.update(RUN_DIR=run_dir, MARTINI_DIR=MARTINI_DIR, REBUILD=REBUILD, TEMP=str(T), BOX_XY=str(BOX_XY),
                  INSANE_LIPIDS=' '.join(f'-l {item}' for item in LIPIDS.split()),
                  START_DIR=f'{local_path}/{START_FROM}' if START_FROM else '',
                  NSTEPS=str(int(SIM_NS * 50000)))    # time step of 20 fs
print('simulation', SIM, 'in', run_dir)

### Building, minimising and equilibrating the system

In [ ]:
%%group 2
%%bash
cd "$RUN_DIR"
step() { local log=$1; shift; if ! "$@" > "$log" 2>&1; then echo "ERROR: last lines of $log"; tail -25 "$log"; exit 1; fi; }

if [ "$REBUILD" = "yes" ]; then rm -f eq.* md.* min.* \#*; fi
if [ -f eq.gro ]; then
  echo "This simulation has already been prepared: go to the production cell."
  echo "To start it again from scratch set REBUILD = 'yes' in the parameters cell."
  exit 0
fi
cp $MARTINI_DIR/*.itp .
# parameters at the temperature of this simulation
for f in martini_eq martini_md; do
  sed -e "s/^ref_t .*/ref_t                    = $TEMP/" -e "s/^gen_temp .*/gen_temp                 = $TEMP/" \
      -e "s/^nsteps .*/nsteps                   = $NSTEPS/" $MARTINI_DIR/$f.mdp > $f.mdp
done
sed -i "s/^nsteps .*/nsteps                   = 100000/" martini_eq.mdp       # 1 ns of equilibration

if [ -z "$START_DIR" ]; then
  # a new bilayer with insane: same composition in both leaflets, 150 mM NaCl
  step insane.log python3 $MARTINI_DIR/insane.py $INSANE_LIPIDS -x $BOX_XY -y $BOX_XY -z 12 -sol W -salt 0.15 -o bilayer.gro -p insane.top
  cp $MARTINI_DIR/martini_lipids.top topol.top
  sed -n '/\[ system \]/,$p' insane.top >> topol.top
  step grompp_min.log gmx grompp -f $MARTINI_DIR/martini_em.mdp -c bilayer.gro -p topol.top -o min.tpr
  step min.log gmx mdrun -deffnm min
  start=min.gro
else
  # the last frame of another simulation (e.g. a gel, to heat it)
  [ -f "$START_DIR/md.gro" ] || { echo "ERROR: $START_DIR/md.gro not found: has that simulation finished?"; exit 1; }
  cp "$START_DIR/md.gro" start.gro
  cp "$START_DIR/topol.top" topol.top
  start=start.gro
fi
sed -n '/molecules/,$p' topol.top
step grompp_eq.log gmx grompp -f martini_eq.mdp -c $start -p topol.top -o eq.tpr
step eq.log gmx mdrun -deffnm eq
step grompp_md.log gmx grompp -f martini_md.mdp -c eq.gro -t eq.cpt -p topol.top -o md.tpr
echo "Done: simulation ready"

### Production

In [ ]:
%%group 2
%cd {run_dir}
!gmx mdrun -deffnm md -cpi md.cpt -cpt 5 -v

### Post-processing

This cell makes the molecules whole for visualisation and writes a PDB file with the bonds (open `md_whole.pdb` and load `md_whole.xtc` into it with VMD or ChimeraX), removes the jumps across the box for the diffusion, and computes the density profiles.

In [ ]:
%%group 2
%%bash
cd "$RUN_DIR"
step() { local log=$1; shift; if ! "$@" > "$log" 2>&1; then echo "ERROR: last lines of $log"; tail -25 "$log"; exit 1; fi; }
echo System | step trjconv.log gmx trjconv -s md.tpr -f md.xtc -pbc mol -o md_whole.xtc
echo System | step nojump.log gmx trjconv -s md.tpr -f md.xtc -pbc nojump -o md_nojump.xtc
echo System | step dump.log gmx trjconv -s md.tpr -f md_whole.xtc -dump 0 -o md_first.gro
step energy.log bash -c 'printf "Box-X\nBox-Y\nBox-Z\n\n" | gmx energy -f md.edr -o box.xvg'
# lateral diffusion of the lipids (phosphate beads) and diffusion of water
step msd.log gmx msd -s md.tpr -f md_nojump.xtc -sel "name PO4" -lateral z -trestart 500 -beginfit 2000 -endfit 20000 -o msd_PO4.xvg
step msd_water.log gmx msd -s md.tpr -f md_nojump.xtc -sel "name W" -trestart 500 -beginfit 2000 -endfit 20000 -o msd_W.xvg
grep 'legend "D' msd_PO4.xvg msd_W.xvg | cut -d'"' -f2
echo "If the diffusion coefficient of water is close to zero, the water has frozen: tell the teacher."

In [ ]:
%%group 2
import MDAnalysis as mda
u = mda.Universe(f'{run_dir}/md.tpr', f'{run_dir}/md_whole.xtc')
u.select_atoms('all').write(f'{run_dir}/md_whole.pdb', bonds='conect')
data, _ = read_xvg(f'{run_dir}/box.xvg')
n_heads = len(u.select_atoms('name PO4 ROH'))
plt.plot(data[:, 0] / 1000, data[:, 1] * data[:, 2] / (n_heads / 2))
plt.xlabel('Time (ns)')
plt.ylabel('Area per molecule (nm$^2$)')
plt.title(SIM)
plt.show()

# GROUP 2, Part 2: the report

The analysis cells read all the simulations that are in the group folder, so they can be run by any member once the simulations are finished.

## 1. Introduction

*Describe the phases of lipid bilayers, the main transition, the effect of cholesterol and what is known from experiments. What is the Martini model and what do you expect from it?*

## 2. Methods

*Describe the systems (composition, number of lipids, water and ions), the force field, the simulation protocol (temperatures, pressure coupling, time step, length) and the analyses.*

In [ ]:
%%group 2
import re
import MDAnalysis as mda

sims = {os.path.basename(d): d for d in sorted(glob.glob(f'{local_path}/*K*')) if os.path.exists(f'{d}/md_nojump.xtc')}
print('finished simulations:', list(sims))

def tail_chains(u, lipid):
    # the beads along the two tails of a phospholipid, from the glycerol to the end
    names = list(u.select_atoms(f'resname {lipid}').residues[0].atoms.names)
    chains = []
    for glycerol, t in [('GL1', 'A'), ('GL2', 'B')]:
        beads = sorted([n for n in names if re.fullmatch(f'[CD][0-9]{t}', n)], key=lambda n: int(n[1]))
        chains.append([glycerol] + beads)
    return chains

def leaflet_side(heads, box):
    # +1 for the head beads above the centre of the membrane, -1 for those below (periodic along z)
    z, lz = heads.positions[:, 2], box[2]
    angle = 2 * np.pi * z / lz
    center = lz * np.arctan2(np.sin(angle).mean(), np.cos(angle).mean()) / (2 * np.pi)
    dz = (z - center + lz / 2) % lz - lz / 2
    return np.sign(dz), dz

def bilayer_properties(d, skip=1):
    # time series for simulation folder d: time (ns), area per molecule (nm^2), P-P thickness (nm),
    # average P2 of the phospholipid tails and P2 of each phospholipid (frames x lipids)
    u = mda.Universe(f'{d}/md.tpr', f'{d}/md.xtc')
    phospholipids = sorted(set(u.select_atoms('name PO4').resnames))
    heads = u.select_atoms('name PO4')
    all_heads = u.select_atoms('name PO4 ROH')           # phospholipids and cholesterol
    bonds = {}                                           # {lipid: [(beads a, beads b), ...]} along the tails
    for lipid in phospholipids:
        bonds[lipid] = [(u.select_atoms(f'resname {lipid} and name {a}'), u.select_atoms(f'resname {lipid} and name {b}'))
                        for chain in tail_chains(u, lipid) for a, b in zip(chain[:-1], chain[1:])]
    out = {'time': [], 'area': [], 'thickness': [], 'p2': [], 'p2_lipid': []}
    for ts in u.trajectory[::skip]:
        side, dz = leaflet_side(heads, ts.dimensions)
        out['time'].append(ts.time / 1000)
        out['area'].append(ts.dimensions[0] * ts.dimensions[1] / 100 / (len(all_heads) / 2))
        out['thickness'].append((dz[side > 0].mean() - dz[side < 0].mean()) / 10)
        p2_lipid = []
        for lipid in phospholipids:
            p2 = []
            for a, b in bonds[lipid]:
                v = b.positions - a.positions
                v -= ts.dimensions[:3] * np.round(v / ts.dimensions[:3])
                p2.append(1.5 * (v[:, 2] / np.linalg.norm(v, axis=1)) ** 2 - 0.5)
            p2_lipid.append(np.mean(p2, axis=0))          # average over the bonds of each lipid
        p2_lipid = np.concatenate(p2_lipid)
        out['p2'].append(p2_lipid.mean())
        out['p2_lipid'].append(p2_lipid)
    return {k: np.array(v) for k, v in out.items()}

props = {name: bilayer_properties(d) for name, d in sims.items()}

## 3. Results

### 3.1 Time series and equilibration

*Look at the area per molecule, the thickness and the order of the tails as a function of time. Do they reach a plateau? How long does it take? Which part of the simulations can you use for the averages? Look at the trajectories too: what happens in the runs where the properties change?*

The fraction of *ordered lipids* (those whose tails have an average $P_2$ above a threshold, here 0.7: a rough criterion, since the order of single lipids fluctuates a lot) helps to follow the formation of a gel, which starts from a small ordered nucleus that grows.

In [ ]:
%%group 2
P2_ORDERED = 0.7
fig, axes = plt.subplots(2, 2, figsize=(16, 9))
colors = dict(zip(sims, plt.cm.tab20.colors))
for name, p in props.items():
    axes[0, 0].plot(p['time'], p['area'], color=colors[name], label=name)
    axes[0, 1].plot(p['time'], p['thickness'], color=colors[name])
    axes[1, 0].plot(p['time'], p['p2'], color=colors[name])
    axes[1, 1].plot(p['time'], (p['p2_lipid'] > P2_ORDERED).mean(axis=1), color=colors[name])
axes[0, 0].set_ylabel('Area per molecule (nm$^2$)')
axes[0, 1].set_ylabel('$D_{PP}$ (nm)')
axes[1, 0].set_ylabel('Average $P_2$ of the tails')
axes[1, 1].set_ylabel('Fraction of ordered lipids')
for ax in axes.flat:
    ax.set_xlabel('Time (ns)')
axes[0, 0].legend(fontsize=8)
plt.tight_layout()
plt.show()

### 3.2 The main transition of DPPC

*Plot the averages of the area per lipid, the thickness and the order parameter as a function of temperature, for the cooling and the heating runs. Where is the transition? Is it sharp? Is there hysteresis, and why? Compare the area per lipid and the thickness in the fluid phase with the experimental data (`Data/martini/benchmark`) and the transition temperature with the experimental one.*

In [ ]:
%%group 2
SKIP_NS = 50          # initial part of each production run excluded from the averages: check it with the time series

rows = []
for name, p in props.items():
    keep = p['time'] > SKIP_NS
    temperature = int(re.search(r'_(\d+)K', name).group(1))
    rows.append({'simulation': name, 'T (K)': temperature, 'heating': '_from_' in name,
                 'area (nm2)': p['area'][keep].mean(), 'D_PP (nm)': p['thickness'][keep].mean(),
                 'P2': p['p2'][keep].mean(), 'ordered fraction': (p['p2_lipid'][keep] > P2_ORDERED).mean()})
table = pd.DataFrame(rows)
display(table.round(3))

In [ ]:
%%group 2
# your code here

### 3.3 Lateral diffusion

*The post-processing cell computed with `gmx msd` the lateral diffusion coefficient of the lipids (phosphate beads) and the diffusion coefficient of water (files `msd_PO4.xvg` and `msd_W.xvg` in each folder; the fitted value is in the legend, `read_xvg(...)[1][0]`). Check that the fitting interval (`-beginfit`, `-endfit`) is in the linear part of the MSD. How does the diffusion of the lipids change at the transition and with cholesterol? Compare with experimental values and discuss the time scale of the Martini model. Check also that the water never froze.*

In [ ]:
%%group 2
# your code here

### 3.4 The effect of cholesterol

*How do the area per molecule, the thickness and the order of the DPPC tails change with the cholesterol concentration, at 323 K and below the transition of pure DPPC? Does cholesterol remove the transition? What happens to the diffusion? Compare with experiments (for example the condensing effect measured by X-ray scattering and NMR).*

In [ ]:
%%group 2
# your code here

### 3.5 Structure of the bilayers

*Compare the density profiles (`gmx density`, see the training notebook) and pictures of a fluid bilayer, a gel and a bilayer with cholesterol. What is the tilt of the tails in the gel? Where is cholesterol located?*

In [ ]:
%%group 2
# your code here

## 4. Discussion

*Interpret the results. Some points to consider: How well does Martini reproduce the phase behaviour of DPPC and the effect of cholesterol? What are the effects of the size of the system and of the length of the simulations on the transition you observed? What does the comparison of the diffusion coefficients tell you about the dynamics in coarse-grained models?*

## 5. Conclusions

## References

*   Souza PCT et al. (2021) Martini 3: a general purpose force field for coarse-grained molecular dynamics. *Nat Methods* 18:382
*   Pedersen KB et al. (2025) The Martini 3 lipidome: expanded and refined parameters improve lipid phase behavior. *ACS Cent Sci*, https://doi.org/10.1021/acscentsci.5c00755
*   Borges-Araújo L et al. (2023) Martini 3 coarse-grained force field for cholesterol. *J Chem Theory Comput* 19:7387
*   Wassenaar TA et al. (2015) Computational lipidomics with insane. *J Chem Theory Comput* 11:2144
*   Kučerka N, Nieh MP, Katsaras J (2011) Fluid phase lipid areas and bilayer thicknesses of commonly used phosphatidylcholines as a function of temperature. *Biochim Biophys Acta* 1808:2761
*   *add the experimental papers you used*